In [0]:
from pyspark.sql.functions import col
# 1. Identify a default dataset: 'samples.nyctaxi.trips' is available in Unity Catalog
source_table = "samples.nyctaxi.trips"

# 2. Consume the data using structured streaming
streaming_df = spark.readStream.table(source_table)

# 3. Write output files to a directory in Delta format
output_path = "/Volumes/workspace/default/my_data_volume/nyctaxi_stream_output"
checkpoint_path_10001 = "/Volumes/workspace/default/my_data_volume/nyctaxi_stream_checkpoint_10001"
checkpoint_path_10044 = "/Volumes/workspace/default/my_data_volume/nyctaxi_stream_checkpoint_10044"

streaming_query_10001 = streaming_df.filter(col("pickup_zip")=='10001').writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path_10001) \
    .trigger(availableNow=True) \
    .start(output_path)

streaming_query_10044 = streaming_df.filter(col("pickup_zip")=='10044').writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path_10044) \
    .trigger(availableNow=True) \
    .start(output_path)

In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/my_data_volume/nyctaxi_stream_output")
display(df.groupBy("pickup_zip").count())


In [0]:
# stop streaming
for stream in spark.streams.active:
  # stream.stop()
  # stream.awaitTermination()
  print(stream)

In [0]:
streaming_df.printSchema()

In [0]:
streaming_query_10044.status

In [0]:
%sql
DESCRIBE HISTORY '/Volumes/workspace/default/my_data_volume/nyctaxi_stream_output'


In [0]:
%sql
SHOW TABLES IN workspace.default

In [0]:
# Read Delta Share table (samples.nyctaxi.trips)
delta_share_df = spark.read.table("samples.nyctaxi.trips")

# Read from S3 (direct URI, not mount)
s3_df = spark.read.format("parquet").load("s3a://your-bucket/path/to/data/")

# Read from Azure Blob Storage Gen2 (direct URI, not mount)
azure_df = spark.read.format("parquet").load("abfss://your-container@your-account.dfs.core.windows.net/path/to/data/")

# Example: Union all dataframes (assuming schemas match)
combined_df = delta_share_df.unionByName(s3_df).unionByName(azure_df)
display(combined_df)

## FROM LOCAL only via bloblike
```
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DeltaLocal").getOrCreate()
df = spark.read.format("delta").load("s3a://your-bucket/path/to/delta-table")
df.show()
```
## Or hard way

Yes, you can use Unity Catalog credential vending to access Delta tables managed by Databricks from external clients, including local scripts. This allows your local Delta Lake or Iceberg client to inherit privileges and securely access data governed by Databricks
1
. You must configure your client to use the credentials provided by Databricks, typically via REST API or credential files.

If you are accessing a Delta table shared via Delta Sharing, you can use a credential file (such as a .share profile) to authenticate and read the shared data from your local environment using the delta-sharing Python connector
2
.

For Unity Catalog managed tables, refer to Databricks documentation for configuring credential vending and REST API access
1
.


In [0]:
%sql
DESCRIBE EXTENDED workspace.default.feature_demo_table

In [0]:

spark.conf.get("spark.sql.shuffle.partitions")

In [0]:

df = spark.createDataFrame(
    [
        ("sue", 32),
        ("li", 3),
        ("bob", 75),
        ("heo", 13),
    ],
    ["first_name", "age"],
)


In [0]:
display(df)

In [0]:
from pyspark.sql.functions import col, when

df1 = df.withColumn(
    "life_stage",
    when(col("age") < 13, "child").when(col("age").between(13, 19), "teenager")
    .otherwise("adult"),
)

In [0]:
%sql 
describe extended some_people

In [0]:
df1.write.saveAsTable("some_people")

In [0]:
df.explain()

In [0]:
%sql
DESCRIBE EXTENDED '/Volumes/workspace/default/my_data_volume/nyctaxi_stream_output'

In [0]:
%sql
CREATE TABLE workspace.default.nytaxi_stream_output
USING DELTA
LOCATION '/Volumes/workspace/default/my_data_volume/nyctaxi_stream_output'

In [0]:
%pip install dbdemos -U

In [0]:
import dbdemos
dbdemos.install('declarative-pipeline-cdc')